# Reinforcement Learning in Bomberman: Comparing Q-Learning and SARSA with Linear Function Approximation

**Author:** Samira Mohammadi  
**Course:** Machine Learning Essentials — Summer Semester 2026, Heidelberg University  
**Code Repository:** [https://github.com/Samira/bomberman-rl-project]  
**Dependencies:** `python >= 3.8`, `numpy`, `torch`, `scipy`, `matplotlib`, `pygame`, `tqdm`

---

## 1. Introduction
*Author: Samira Mohammadi*

The goal of this project is to develop and train an autonomous agent for the arcade game Bomberman using reinforcement learning within the provided BombeRLe framework. The game takes place on a discrete $17 \times 17$ grid where four agents compete simultaneously. Each step, an agent can move, wait, or place a bomb. Bombs destroy wooden crates, uncover coins, and eliminate opponents, but they also pose a direct threat to the agent itself.

Training an agent in this environment presents three main challenges:
* **Delayed hazards:** When a bomb is dropped, it ticks for four steps before exploding, and the blast remains lethal for another step. The agent cannot act purely reactively; it must plan escape routes ahead of time to avoid getting trapped in blast zones.
* **Sparse rewards:** In the base game, points are only awarded for picking up coins ($+1$) and eliminating opponents ($+5$). An agent exploring purely by chance rarely drops a useful bomb and escapes safely, making learning without additional feedback very slow.
* **Dynamic opponents:** Opponents actively modify the terrain by destroying crates and placing bombs, creating an unpredictable and non-stationary environment.

Instead of training a computationally heavy deep neural network that might struggle to converge in time or exceed the strict step execution limit of $0.5\text{ seconds}$, I focus on linear function approximation. Using hand-crafted features based on breadth-first search (BFS) for pathfinding and danger detection, I implement and compare two core algorithms from the lecture: Linear Q-Learning (off-policy) and Linear SARSA (on-policy). This report describes the feature design, reward shaping strategy, training process, and a systematic experimental evaluation of both agents against the baseline opponents.

## 2. Background
*Author: Samira Mohammadi*

Reinforcement learning provides the mathematical framework for our Bomberman setup. The environment is framed as a Markov Decision Process (MDP) with the standard tuple:

$$\mathcal{M} = (\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$$

Here, $\mathcal{S}$ is the environment state space, $\mathcal{A}$ is the action space, $\mathcal{P}$ represents the transition probabilities $P(s_{t+1} \mid s_t, a_t)$, $\mathcal{R}$ is the immediate reward function, and $\gamma \in [0, 1)$ is the discount factor. At every tick, the agent chooses from six discrete actions:

$$\mathcal{A} = \{\text{'UP'}, \text{'DOWN'}, \text{'LEFT'}, \text{'RIGHT'}, \text{'WAIT'}, \text{'BOMB'}\}$$

### 2.1 Tabular Limitations and Linear Function Approximation
In Bomberman, a single raw state $s \in \mathcal{S}$ tracks a $17 \times 17$ board. This includes stone walls, destructible crates, free coins, bomb timers, blast areas, and the coordinates of all four agents.

Because of all these moving parts, the total number of configurations easily exceeds $10^{20}$ states:

$$\vert{}\mathcal{S}\vert{} \gg 10^{20}$$

A tabular approach like standard Q-learning fails here. The agent visits practically every board layout only once across 300 rounds. A lookup table cannot generalize this experience to new, unseen board layouts.

To make learning feasible, I implemented linear function approximation. Instead of storing values for individual states, the model approximates the action-value function $Q(s, a)$ as a weighted sum of extracted features:

$$\hat{Q}(s, a; \mathbf{w}) = \mathbf{w}^\top \boldsymbol{\phi}(s, a) = \sum_{i=1}^{d} w_i \phi_i(s, a)$$

Here, $\boldsymbol{\phi}(s, a) \in \mathbb{R}^d$ is a feature vector built from domain knowledge, and $\mathbf{w} \in \mathbb{R}^d$ is the weight vector updated during training. This setup fits our constraints well:
* **Low Latency:** Computing $\mathbf{w}^\top \boldsymbol{\phi}(s, a)$ takes under a millisecond. This easily satisfies the tournament limit of $0.5\text{ seconds}$ per turn.
* **Simple Optimization:** Linear models do not suffer from the local minima problems common in deep networks, making convergence behavior easier to track.

### 2.2 Temporal Difference Control Algorithms
I compared two basic temporal difference methods in `learner.py`: off-policy Q-Learning and on-policy SARSA.

#### Linear Q-Learning (Off-Policy TD Control)
Q-Learning targets the optimal action-value function directly. When computing the update for a step $(s_t, a_t, R_{t+1}, s_{t+1})$, it always selects the greedy action in the next state, regardless of whether the agent explored or exploited:

$$\delta_t = R_{t+1} + \gamma \max_{a' \in \mathcal{A}} \hat{Q}(s_{t+1}, a'; \mathbf{w}_t) - \hat{Q}(s_t, a_t; \mathbf{w}_t)$$

The weights update using semi-gradient descent:

$$\mathbf{w}_{t+1} = \mathbf{w}_t + \alpha \delta_t \nabla_{\mathbf{w}} \hat{Q}(s_t, a_t; \mathbf{w}_t) = \mathbf{w}_t + \alpha \delta_t \boldsymbol{\phi}(s_t, a_t)$$

where $\alpha$ is the learning rate. Because the $\max$ operator assumes optimal play downstream, Q-Learning learns aggressive paths toward coins and crates. However, it can overestimate state values and does not account for self-inflicted damage caused by random exploratory moves during training.

#### Linear SARSA (On-Policy TD Control)
SARSA instead evaluates the actual policy being followed. The target calculation uses the next action $a_{t+1}$ that was actually sampled from the $\epsilon$-greedy policy:

$$\delta_t = R_{t+1} + \gamma \hat{Q}(s_{t+1}, a_{t+1}; \mathbf{w}_t) - \hat{Q}(s_t, a_t; \mathbf{w}_t)$$

The weight update rule remains mathematically identical:

$$\mathbf{w}_{t+1} = \mathbf{w}_t + \alpha \delta_t \boldsymbol{\phi}(s_t, a_t)$$

Because SARSA considers exploratory moves in its updates, it penalizes actions that bring the agent near bombs. If an exploratory action could result in stepping into a blast zone, SARSA factors that risk into the current value. As a result, the agent learns a safer, more defensive policy during training.

## 3. Project Planning
*Author: Samira Mohammadi*

Since this project was carried out as a single-author submission, I organized the development into three main phases to meet both the code submission deadline (September 21) and the final report deadline (September 28):

* **Phase 1: Setup and Feature Design (Sep 10 – Sep 13):**  
  I set up the local repository, verified the game environment using the official Dockerfile, and designed the initial feature representation. The main focus was implementing the breadth-first search (BFS) helper functions to track safe tiles, danger zones, and the nearest targets.

* **Phase 2: Curriculum Training and Validation (Sep 14 – Sep 17):**  
  Training started on simpler tasks following the project guidelines (Task 1 in `coin-heaven` for pure navigation, followed by Task 2 in `classic` for crate clearing). I also configured the auxiliary rewards to prevent self-destruction. On September 17, I uploaded a test archive to MaMPF to verify that the agent runs without errors in the university evaluation environment.

* **Phase 3: Comparison, Evaluation, and Documentation (Sep 18 – Sep 28):**  
  I implemented the second agent (Linear SARSA) to compare its on-policy behavior against Linear Q-Learning. Both models were trained and benchmarked against the baseline agents (`rule_based_agent` and `random_agent`) across multiple rounds to gather quantitative performance metrics. After freezing and submitting the final Q-learning agent code on September 21, I focused on compiling the experimental data and writing this report.

All experiments were executed locally on an x86-64 single-core CPU configuration to ensure compliance with the tournament execution environment. Code and weight files were tracked with Git and uploaded to the public GitHub repository prior to report submission.

## 4. Methods
*Author: Samira Mohammadi*

Designing an autonomous Bomberman agent with linear function approximation introduces a fundamental engineering tradeoff: the representation must be computationally lightweight to comfortably clear the strict 0.5-second runtime limit, yet expressive enough to capture spatial dynamics, bomb mechanics, and tactical risks. Because a linear model cannot learn non-linear feature interactions on its own, the feature engineering pipeline serves as the core intelligence of the agent.

### 4.1 Feature Engineering Pipeline
The agent observes a raw state $s \in \mathcal{S}$ provided by the game engine at every step. This dictionary includes the static $17 \times 17$ map, crate coordinates, coin positions, active bombs with their remaining countdown ticks, exploding flame coordinates, and player positions.

To transform this spatial matrix into a compact representation, I implemented an 8-dimensional state feature vector $\boldsymbol{\phi}(s) \in \mathbb{R}^8$ in `callbacks.py`. Computing these features relies heavily on Breadth-First Search (BFS) graph traversals and a predictive hazard matrix.

#### Hazard Projection and Threat Detection
Before evaluating movement decisions, the agent constructs a spatial hazard map $\mathcal{H} \in \{0, 1, 2, 3, 4\}^{17 \times 17}$. For every ticking bomb located at $(x_b, y_b)$ with remaining timer $\tau \in \{0, 1, 2, 3\}$, the blast footprint expands horizontally and vertically up to a range of 3 tiles:

$$\text{Blast}((x_b, y_b)) = \{(x_b \pm k, y_b) \mid 0 \le k \le 3\} \cup \{(x_b, y_b \pm k) \mid 0 \le k \le 3\}$$

The expansion along any axis halts immediately if it encounters an indestructible wall. If an explosion is currently active ($\tau = 0$), the affected tiles are marked as lethal.

Using this hazard matrix, the first five features capture direct spatial safety:
* **Current Danger Tile ($\phi_1$):** A binary indicator set to $1.0$ if the tile currently occupied by the agent $(x, y)$ is marked in $\mathcal{H}$ as an active explosion or within an imminent blast path ($\tau \le 3$), and $0.0$ if the tile is completely safe.
* **Directional Hazards ($\phi_2, \phi_3, \phi_4, \phi_5$):** Four separate directional features corresponding directly to the cardinal movements $[\text{'UP'}, \text{'DOWN'}, \text{'LEFT'}, \text{'RIGHT'}]$. Each feature $\phi_d$ evaluates the immediate adjacent coordinate $(x_d, y_d)$:
  $$\phi_d = \begin{cases} 1.0 & \text{if } (x_d, y_d) \text{ is blocked by a wall, crate, opponent, or lies within } \mathcal{H} \\ 0.0 & \text{if the step is unobstructed and safe} \end{cases}$$
  These inputs allow the linear weights to quickly learn heavy negative associations with blocked or fatal directions.

#### Breadth-First Search for Navigation and Escape
For high-level navigation, simple Manhattan distance fails because stone walls and crates create complex dead ends. I therefore deployed a queue-based Breadth-First Search (BFS) to compute true shortest walkable paths on the grid graph $G = (V, E)$.

The algorithm starts from the agent's current position and explores neighboring walkable coordinates while avoiding active explosion zones.
* **Direction to Nearest Target ($\phi_6$):** The search prioritizes free coins first. If coins exist on the board and are reachable, BFS finds the shortest path to the nearest coin and identifies the direction of the very first step required:
  $$\text{Target Direction} \in \{\text{UP}, \text{DOWN}, \text{LEFT}, \text{RIGHT}, \text{NONE}\}$$
  This categorical step is mapped to an index and normalized:
  $$\phi_6 = \frac{\text{StepIndex}}{4.0} \in [0.0, 1.0]$$
  If all coins have been collected or blocked off, the search target switches to the closest destructible crate adjacent to a walkable tile, allowing the agent to transition naturally from collecting to clearing obstacles.
* **Safe Bomb Deployment Flag ($\phi_7$):** One of the primary failure modes in early experiments was suicidal bombing, where the agent dropped a bomb at the end of a corridor with nowhere to run. Feature $\phi_7$ acts as a strict safety guard:
  $$\phi_7 = \begin{cases} 1.0 & \text{if bomb ready} \land (\text{adjacent crate or opponent}) \land (\text{BFS finds a safe tile with distance } \le \tau_{\text{bomb}}) \\ 0.0 & \text{otherwise} \end{cases}$$
  The feature only turns positive if dropping a bomb damages an objective and the BFS explicitly confirms the existence of an accessible escape tile outside the blast footprint before the 4-tick detonation timer expires.
* **Corridor Escape Ratio ($\phi_8$):** Measures local maneuverability by counting how many immediate neighbors are free and safe, divided by 4:
  $$\phi_8 = \frac{1}{4} \sum_{d \in \{\text{U, D, L, R}\}} \mathbb{I}(\text{Tile}_d \text{ is free and safe})$$
  This gives the model a continuous metric indicating whether it is moving into an open area or entering a narrow dead-end corridor.

| Index | Feature Name | Value Range | Primary Strategic Role |
| :---: | :--- | :---: | :--- |
| $\phi_1$ | Current Danger | $\{0.0, 1.0\}$ | Immediate emergency evasion |
| $\phi_2$ | Up Obstacle/Danger | $\{0.0, 1.0\}$ | Collision and explosion avoidance |
| $\phi_3$ | Down Obstacle/Danger | $\{0.0, 1.0\}$ | Collision and explosion avoidance |
| $\phi_4$ | Left Obstacle/Danger | $\{0.0, 1.0\}$ | Collision and explosion avoidance |
| $\phi_5$ | Right Obstacle/Danger | $\{0.0, 1.0\}$ | Collision and explosion avoidance |
| $\phi_6$ | Target Direction | $[0.0, 1.0]$ | BFS pathfinding toward coins and crates |
| $\phi_7$ | Safe Bomb Viability | $\{0.0, 1.0\}$ | Controlled demolition without self-trapping |
| $\phi_8$ | Mobility Ratio | $[0.0, 1.0]$ | Spatial awareness and open-space preference |

### 4.2 State-Action Feature Mapping
To allow the linear model to assign distinct valuations to different actions rather than evaluating only the passive board state, I expanded the state representation into a structured block-sparse matrix.

For an action space containing $\vert{}\mathcal{A}\vert{} = 6$ actions, the state feature vector $\boldsymbol{\phi}(s) \in \mathbb{R}^8$ is mapped into a combined state-action feature vector $\boldsymbol{\Phi}(s, a) \in \mathbb{R}^{48}$:

$$\boldsymbol{\Phi}(s, a) = \begin{bmatrix} \mathbf{0} \\ \vdots \\ \boldsymbol{\phi}(s) \\ \vdots \\ \mathbf{0} \end{bmatrix} \leftarrow \text{slot corresponding to action } a$$

Under this block formulation, the global weight vector $\mathbf{w} \in \mathbb{R}^{48}$ is conceptually partitioned into six independent 8-dimensional weight vectors $\mathbf{w} = [\mathbf{w}_{\text{UP}}^\top, \mathbf{w}_{\text{DOWN}}^\top, \mathbf{w}_{\text{LEFT}}^\top, \mathbf{w}_{\text{RIGHT}}^\top, \mathbf{w}_{\text{WAIT}}^\top, \mathbf{w}_{\text{BOMB}}^\top]^\top$. The inner product computation simplifies neatly:

$$\hat{Q}(s, a; \mathbf{w}) = \mathbf{w}^\top \boldsymbol{\Phi}(s, a) = \mathbf{w}_a^\top \boldsymbol{\phi}(s)$$

This decouples the parameter updates during training. Taking action $a_t$ modifies only the weight slice $\mathbf{w}_{a_t}$, preventing updates for one action (e.g., dropping a bomb) from unintentionally distorting the estimated values of independent actions (e.g., evading left).

### 4.3 Reward Shaping and Mitigation of Pathological Behaviors
The default game environment provides extremely sparse terminal feedback: positive rewards are awarded only when an agent actually steps onto a coin tile ($+1$) or eliminates an opponent ($+5$), while dying yields a large penalty ($-5$). Relying purely on sparse feedback led to unguided random walks where the agent rarely found coins before dying of turn timeouts.

To provide a consistent learning signal, I engineered auxiliary intermediate rewards in `train.py`. However, uncalibrated intermediate rewards frequently create policy loops. During initial training runs, giving $+0.2$ for simply moving toward a coin caused the agent to move toward the coin, step backward, and step forward again to farm infinite rewards.

To resolve this issue, I introduced asymmetric penalties and step costs:
* **Target Distance Tracking:** When the agent takes an action that strictly decreases the BFS path distance to the nearest target, it receives $+0.3$. Moving away or oscillating incurs a stiffer penalty of $-0.35$. The $-0.05$ difference acts as a negative step tax, breaking cyclic back-and-forth loops.
* **Blast Evacuation Reward:** Successfully transitioning from a danger tile ($\phi_1 = 1.0$) into a verified safe tile yields $+0.5$. In contrast, walking into an active blast zone incurs $-2.0$.
* **Safe Demolition Incentives:** Executing `'BOMB'` when $\phi_7 = 1.0$ yields $+0.8$. Dropping a bomb without a confirmed escape route or without adjacent crates yields a harsh penalty of $-1.5$, teaching the model that uncalculated bombings are actively harmful.
* **Invalid Action Suppression:** Attempting to move into indestructible walls or crates triggers the game's `e.INVALID_ACTION` event, which is penalized with $-0.1$. Unproductive waiting (`'WAIT'`) in safe situations is penalized with $-0.05$ to encourage proactive map exploration.

## 5. Training
*Author: Samira Mohammadi*

Training was carried out using a three-stage curriculum so that the agent could learn foundational navigation and safe bomb handling before facing competitive opponents:

* **Stage 1: Navigation and Coin Collection:**  
  In the `coin-heaven` scenario (without crates or bombs), the agent trained for 100 rounds to learn efficient movement toward goals. The exploration rate $\epsilon$ decayed linearly from $1.0$ down to $0.1$.

* **Stage 2: Obstacle Demolition and Escape:**  
  In the `classic` scenario (crates present, no opponents), the agent trained for 300 rounds. The focus was on learning to deploy bombs next to crates while escaping the blast radius safely.

* **Stage 3: Multi-Agent Competition:**  
  Finally, the agent trained for 300 rounds in the full `classic` scenario against one `rule_based_agent` and two instances of `random_agent`. The exploration rate decayed toward $\epsilon = 0.05$ to stabilize policy behavior under competitive conditions.

### 5.1 Hyperparameters
Weight updates were performed online at each step transition and round end using semi-gradient descent. The learning parameters were tuned to balance rapid feature association with numerical stability against sudden death penalties. The final hyperparameters configured in `learner.py` are summarized below:

| Hyperparameter | Symbol | Value | Configuration Rationale |
| :--- | :---: | :---: | :--- |
| Learning Rate | $\alpha$ | $0.005$ | Small step size preventing weight divergence from severe suicide penalties |
| Discount Factor | $\gamma$ | $0.95$ | Prioritizes multi-step planning and safe blast escape horizons |
| Initial Exploration | $\epsilon_{\text{start}}$ | $0.20$ | Controlled exploration around domain-engineered BFS features |
| Minimum Exploration | $\epsilon_{\text{end}}$ | $0.05$ | Sustains residual exploration while evaluating converged policies |
| Training Horizon | $N$ | 300 rounds | Full training run evaluated on classic boards |
| Evaluation Horizon | $M$ | 50 rounds | Benchmark games executed with frozen policies ($\epsilon = 0$) |

#### Parameter Sensitivity and Calibration
* **Learning Rate ($\alpha = 0.005$):** Because stepping into an active blast zone triggers a harsh penalty ($-2.0$ to $-5.0$), larger learning rates (such as $\alpha \ge 0.05$) led to severe weight destabilization, causing the model to catastrophically unlearn coin-collection weights after a single accidental death. Setting $\alpha = 0.005$ ensured smooth, monotonic parameter convergence.
* **Discount Factor ($\gamma = 0.95$):** A bomb requires 4 discrete ticks to detonate, followed by 1 tick of lethal flames. Setting $\gamma = 0.95$ provided an effective planning horizon exceeding 15 steps ($\frac{1}{1-\gamma} = 20$), allowing the agent to evaluate the downstream consequences of bomb deployment and secure multi-step escape routes.

## 6. Experiments and Results
*Author: Samira Mohammadi*

To rigorously assess the performance of the implemented algorithms, I tracked the learning progression of both the Linear Q-Learning agent and the Linear SARSA agent across training, evaluated them in direct tournament play against baseline agents, and analyzed the influence of the safety features.

### 6.1 Training Convergence
Both agents were trained for 300 rounds in the `classic` scenario. At the end of each round, the game score was appended to a tracking file (`scores.txt`). To reduce high-variance noise between individual rounds, the scores were smoothed using a moving average window of 20 episodes.

![Training Convergence: Q-Learning vs SARSA](comparison_plot.png)

*Figure 1: Training convergence comparing Linear Q-Learning and Linear SARSA over 300 rounds (20-episode moving average).*

* **Linear Q-Learning:** Demonstrates a steep upward trend after round 100 as the exploration factor decays. The agent learns to actively demolish crates and capture revealed coins. Because Q-learning estimates values based on the greedy action in the next state, it pursues aggressive offensive patterns, occasionally suffering from self-elimination in dense blast configurations.
* **Linear SARSA:** Exhibits a smoother and slightly slower score rise. Because SARSA samples the next action from its current $\epsilon$-greedy exploratory behavior, the updates internalize the risk of exploratory steps into blast zones. This produces a more cautious policy that avoids risky narrow corridors.

### 6.2 Tournament Evaluation
After completing training, exploration was disabled ($\epsilon = 0$) to benchmark the frozen policies. Both agents played 50 evaluation games in the `classic` setting alongside the framework's `rule_based_agent` and `random_agent`.

| Agent | Average Score | Win Rate (%) | Suicides |
| :--- | :---: | :---: | :---: |
| **Linear Q-Agent (My Agent)** | 4.10 | 36.0% | 4 |
| **Linear SARSA (My Agent)** | 3.35 | 26.0% | 2 |
| **Rule-Based Agent** | 4.60 | 38.0% | 3 |
| **Random Agent** | 0.30 | 0.0% | 41 |

The Linear Q-Agent achieves a competitive 36% win rate, close to the heuristic rule-based benchmark, by focusing heavily on crate clearing. While SARSA achieves a lower average score (3.35), its survival discipline is confirmed by committing only 2 suicides across 50 matches.

### 6.3 Ablation Study: Impact of BFS Safety Features
To measure the contribution of the BFS-driven escape evaluation ($\phi_7$ and $\phi_8$), I trained an ablation variant of the Q-agent without topological danger checks. 
* Without the safe-escape verification, the agent frequently deployed bombs in dead-end corridors, driving the suicide count above 30% of games.
* Re-enabling the BFS hazard projection restricted bomb deployment strictly to states where an exit path was guaranteed, reducing self-inflicted deaths to under 8% in competitive play.

### 6.4 Failure Mode Analysis and Behavioral Anomalies
Evaluating the agents across competitive rounds revealed several structural challenges inherent to linear function approximation in multi-agent environments. Despite achieving competitive win rates and stable scoring profiles, two primary failure modes were observed during empirical testing:

#### 1. Adversarial Deadlocks and Hallway Congestion
The most pronounced tactical failure occurred when an agent encountered an opponent in a narrow, single-tile corridor. Because the linear feature vector $\boldsymbol{\phi}(s)$ treats other players primarily as impassable obstacles (similar to static stone blocks), it lacks an explicit opponent intent model.

When two agents moved toward each other down a dead-end corridor to contest an intermediate coin, my linear agent often found itself unable to anticipate the opponent's subsequent move. If our agent placed a bomb to eliminate the rival, the adversary would frequently place its own bomb simultaneously. This mutual bomb deployment blocked the single available escape tile behind each agent. Because the linear architecture computes valuations independently per action and cannot project adversary response trees, both agents were routinely eliminated in the resulting double blast. Resolving this limitation would require multi-agent modeling or minimax search layers that explicitly represent the opponent's probable action distribution rather than treating them as stationary collision boundaries.

#### 2. Spatial Oscillation and Policy Looping
During early training iterations prior to the calibration of the asymmetric reward step cost, agents frequently exhibited a pathological behavioral loop known as policy oscillation. When positioned exactly midway between two reachable targets (such as two coins or a coin and a crate) with equal graph distances, the BFS feature would alternate directions across consecutive frames.

In step $t$, the agent would step 'RIGHT' toward Coin A, earning a $+0.3$ target-approach reward. In step $t+1$, moving right placed Coin B at an equivalent distance behind it, causing the BFS feature to recompute and indicate 'LEFT' as an optimal trajectory. The agent would immediately step 'LEFT', receiving another positive reward. This produced an infinite two-tile oscillation loop where the agent farmed intermediate distance rewards without ever collecting either coin, eventually dying to the round's turn limit.

To permanently dismantle this pathological loop, I modified the reward structure in `train.py` by introducing an asymmetric distance penalty:
$$\mathcal{R}_{\text{target}} = \begin{cases} +0.30 & \text{if } d(s_{t+1}, \text{target}) < d(s_t, \text{target}) \\ -0.35 & \text{if } d(s_{t+1}, \text{target}) \ge d(s_t, \text{target}) \end{cases}$$
The net return of a back-and-forth cycle evaluates to $+0.30 - 0.35 = -0.05$. This slight negative step tax penalized vacillation and forced the gradient descent updates to favor decisive, monotonic paths toward a single objective.

## 7. Conclusion and Future Work
*Author: Samira Mohammadi*

In this project, I designed and evaluated two autonomous Bomberman agents using linear function approximation with Q-Learning and SARSA. By combining domain-specific topological features (such as BFS distance calculations and dynamic blast-zone projection) with auxiliary reward shaping, the models successfully learned to navigate, break crates, collect coins, and avoid self-trapping without requiring deep neural architectures. The empirical comparisons verified the operational differences between the algorithms: the Linear Q-Agent adopted an aggressive scoring strategy that achieved a competitive win rate against the rule-based benchmark, whereas Linear SARSA prioritized defensive survival maneuvers.

### 7.1 Future Improvements
While the linear models satisfy the 0.5-second runtime constraint and learn effective survival behaviors, several directions could improve their strategic depth:

1. **Non-Linear Function Approximation (DQN):** Implementing a Deep Q-Network operating directly on the 17x17 grid state representation could capture higher-order board patterns and complex chain reactions that hand-crafted linear features miss.
2. **Opponent Tracking and Trapping:** The current feature pipeline treats adversaries primarily as generic obstacles. Incorporating explicit opponent trajectory prediction would enable deliberate trapping moves rather than opportunistic bomb placement.
3. **Automated Feature Selection:** Utilizing feature regularizers (such as L1 LASSO) or automated feature crosses could help identify the most informative spatial combinations while pruning redundant danger indicators.

### 7.2 Recommendations for Future Course Iterations
Based on experimental development within the BombeRLe environment, two minor modifications could enhance tournament evaluation and debugging for future cohorts:
1. **Asymmetric Tick Penalties for Inaction:** Currently, an agent exceeding the $0.5\text{-second}$ computation window is defaulted to `'WAIT'`. Introducing an explicit warning event in the logger when an agent exhausts more than $80\%$ of its allocated tick budget would assist students in profiling BFS bottleneck routines before tournament deployment.
2. **Dynamic Coin Spawning:** In the classic configuration, coins remain statically bound to initial crate distributions. Introducing late-game dynamic coin drops in prolonged rounds would prevent passive stalemates between highly defensive agents.